## GOLD 02. CARGA TABLAS

## 1. Configuración

In [34]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TFM_Gold_Carga") \
    .getOrCreate()

# Trabajar en UTC para que los cambios de hora no dupliquen ni salten horas
spark.conf.set("spark.sql.session.timeZone", "UTC")


ESQUEMA = "gold"
RUTA_FESTIVOS ="gs://tfm_unir1/festivos/festivos_eventos.csv"
RUTA_ESTACIONES = "gs://tfm_unir1/processed/estaciones/"
RUTA_AIRE ="gs://tfm_unir1/processed/aire/eea/horario/"
RUTA_METEO = "gs://tfm_unir1/processed/aire/gva/"
RUTA_TRAFICO = "gs://tfm_unir1/processed/trafico/espiras/"


# Funciones auxiliares: devuelven el nombre completo de la tabla y su ubicación
def tabla(nombre):
    return f"{ESQUEMA}.{nombre}" if ESQUEMA else nombre


## 2. Carga de dim_tiempo

In [13]:
#spark.read.option("header", True).csv(RUTA_FESTIVOS).createOrReplaceTempView("calendario")

from pyspark.sql import functions as F

# Creamos vista para generar las horas
spark.sql("""
    CREATE OR REPLACE TEMP VIEW horas AS
    SELECT EXPLODE(SEQUENCE(
        TIMESTAMP '2013-01-01 00:00:00',
        TIMESTAMP '2027-12-31 23:00:00',
        INTERVAL 1 HOUR)) AS fecha_hora
""")

# Creamos vista para cargar los festivos
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW calendario 
    USING CSV
    OPTIONS (path '{RUTA_FESTIVOS}', header 'true')
""")

# Carga de dim_tiempo usando como join las views horas y calendario
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("dim_tiempo")}
    SELECT
        CAST(DATE_FORMAT(h.fecha_hora, 'yyyyMMddHH') AS INT) AS id_tiempo,
        h.fecha_hora,
        TO_DATE(h.fecha_hora) AS fecha,
        YEAR(h.fecha_hora) AS anio,
        MONTH(h.fecha_hora) AS mes,
        DAY(h.fecha_hora) AS dia,
        HOUR(h.fecha_hora) AS hora,
        DAYOFWEEK(h.fecha_hora) IN (1,7) AS es_fin_de_semana,
        f.nombre IS NOT NULL AS es_festivo,
        f.nombre AS es_evento,
        e.fecha IS NOT NULL AS es_evento,
        TO_DATE(h.fecha_hora) BETWEEN '2020-03-14' AND '2020-06-21' AS es_confinamiento
    FROM horas h
    LEFT JOIN calendario f ON TO_DATE(h.fecha_hora) = TO_DATE(f.fecha) AND f.tipo = 'festivo'
    LEFT JOIN calendario e ON TO_DATE(h.fecha_hora) = TO_DATE(e.fecha) AND e.tipo = 'evento'

""")

spark.sql(f"SELECT COUNT(*) AS horas, COUNT(DISTINCT id_tiempo) AS ids_unicos FROM {tabla('dim_tiempo')}").show()
    


26/10/24 22:18:51 WARN DAGScheduler: Broadcasting large task binary with size 1307.7 KiB


+------+----------+
| horas|ids_unicos|
+------+----------+
|131472|    131472|
+------+----------+



## 3. Carga de dim_estacion

In [15]:
# Creamos vista para cargar las estaciones desde el fichero parquet
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW estaciones 
    USING PARQUET
    OPTIONS (path '{RUTA_ESTACIONES}')
""")

# Carga de dim_estacion. Agrego dos estaciones faltantes a mano
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("dim_estacion")}
    SELECT id_estacion, codigo_eea, nombre_estacion, tipo_estacion, zona,
        latitud, longitud, altitud, fecha_alta, fecha_baja, activa
        
    FROM estaciones 
    
    UNION ALL
    -- Estaciones solo meteorologicas (GVA)
    SELECT * FROM VALUES
        ('46250049', NULL, 'CONSELLERIA METEO.', 'Meteorológica', 'Urbana',
         NULL, NULL, NULL, NULL, NULL, TRUE),
        ('46250900', NULL, 'NAZARET MET-2', 'Meteorológica', 'Urbana',
         NULL, NULL, NULL, NULL, NULL, TRUE)
    
""")

spark.sql(f"SELECT * FROM {tabla('dim_estacion')} ORDER BY id_estacion").show(truncate=False)    

+-----------+----------+------------------+-------------+---------+--------+--------+-------+----------+----------+------+
|id_estacion|codigo_eea|nombre_estacion   |tipo_estacion|zona     |latitud |longitud|altitud|fecha_alta|fecha_baja|activa|
+-----------+----------+------------------+-------------+---------+--------+--------+-------+----------+----------+------+
|46250030   |ES1239A   |PISTA DE SILLA    |Tráfico      |Urbana   |39.45611|-0.37583|11.0   |1992-07-01|null      |true  |
|46250043   |ES1619A   |VIVERS            |Fondo        |Urbana   |39.47806|-0.36833|11.0   |2002-01-01|null      |true  |
|46250046   |ES1885A   |POLITÈCNIC        |Fondo        |Suburbana|39.4803 |-0.3364 |7.0    |2008-04-04|null      |true  |
|46250047   |ES1912A   |AVD. FRANCIA      |Tráfico      |Urbana   |39.4575 |-0.34278|7.0    |2009-01-01|null      |true  |
|46250048   |ES1926A   |MOLÍ DEL SOL      |Tráfico      |Suburbana|39.4811 |-0.4083 |11.0   |2009-08-06|null      |true  |
|46250049   |nul

## 4. Carga de dim_contaminante

In [18]:
# dim_contaminante

# Carga de dim_contaminante

spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("dim_contaminante")}
    SELECT * FROM VALUES
        (8,'NO2', 'µg/m³', 'Media anual', 40.0, 20.0, 10.0),
        (5,'PM10', 'µg/m³', 'Media anual', 40.0, 20.0, 15.0),
        (6001,'PM2.5', 'µg/m³', 'Media anual', 25.0, 10.0, 5.0), 
        (7,'O3', 'µg/m³', 'Dias al año con media de 8h > 120 µg/m³', 25.0, 18.0, NULL)
    
""")

spark.sql(f"SELECT * FROM {tabla('dim_contaminante')}").show(truncate=False)

+---------------+------+------+---------------------------------------+----------------+----------------+-------------+
|id_contaminante|nombre|unidad|indicador                              |limite_ue_actual|objetivo_ue_2030|guia_oms_2021|
+---------------+------+------+---------------------------------------+----------------+----------------+-------------+
|6001           |PM2.5 |µg/m³ |Media anual                            |25.0            |10.0            |5.0          |
|7              |O3    |µg/m³ |Dias al año con media de 8h > 120 µg/m³|25.0            |18.0            |null         |
|8              |NO2   |µg/m³ |Media anual                            |40.0            |20.0            |10.0         |
|5              |PM10  |µg/m³ |Media anual                            |40.0            |20.0            |15.0         |
+---------------+------+------+---------------------------------------+----------------+----------------+-------------+



- UE actual: Directiva 2008/50/CE.
- UE 2030: Directiva (UE) 2024/2881, que son los objetivos que usa el dashboard.
- OMS 2021: guías de calidad del aire de la OMS.

## 5. Carga de fact_aire

In [26]:
# fact_aire

# Creamos vista para cargar los datos desde el fichero parquet
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW aire_eea 
    USING PARQUET
    OPTIONS (path '{RUTA_AIRE}')
""")

spark.sql(f"""
    SELECT id_punto_medicion,
        REGEXP_EXTRACT(id_punto_medicion, 'SP_([0-9]+)_', 1) AS id_estacion,
        COUNT(*) AS registros
        
    FROM aire_eea
    GROUP BY id_punto_medicion
    ORDER BY id_punto_medicion
""").show(50,truncate=False)

# Carga de fact_aire
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("fact_aire")}
    SELECT
        -- La EEA da las horas en UTC: se pasan a hora de Valencia, como el resto de datos
        CAST(DATE_FORMAT(FROM_UTC_TIMESTAMP(fecha_inicio, 'Europe/Madrid'), 'yyyyMMddHH') AS INT) AS id_tiempo,
        REGEXP_EXTRACT(id_punto_medicion, 'SP_([0-9]+)_', 1)  AS id_estacion,
        codigo_contaminante                                   AS id_contaminante,
        AVG(CAST(concentracion AS DOUBLE))                    AS concentracion
    FROM aire_eea
    WHERE concentracion IS NOT NULL
          AND REGEXP_EXTRACT(id_punto_medicion, 'SP_([0-9]+)_', 1) LIKE '46250%'   -- solo estaciones de Valencia ciudad (excluimos 46102002, 46078004, 46190005, 46244003)
    GROUP BY 1, 2, 3
""")

spark.sql(f"SELECT * FROM {tabla('fact_aire')}").show(truncate=False)

+--------------------+-----------+---------+
|id_punto_medicion   |id_estacion|registros|
+--------------------+-----------+---------+
|ES/SP_46078004_14_6 |46078004   |112876   |
|ES/SP_46078004_8_8  |46078004   |107468   |
|ES/SP_46102002_10_46|46102002   |110253   |
|ES/SP_46102002_14_6 |46102002   |112707   |
|ES/SP_46102002_8_8  |46102002   |108971   |
|ES/SP_46102002_9_46 |46102002   |110249   |
|ES/SP_46190005_14_6 |46190005   |109567   |
|ES/SP_46190005_8_8  |46190005   |106058   |
|ES/SP_46244003_10_46|46244003   |27024    |
|ES/SP_46244003_14_6 |46244003   |70056    |
|ES/SP_46244003_8_8  |46244003   |61296    |
|ES/SP_46244003_9_46 |46244003   |27024    |
|ES/SP_46250030_10_46|46250030   |104325   |
|ES/SP_46250030_14_6 |46250030   |110663   |
|ES/SP_46250030_8_8  |46250030   |110593   |
|ES/SP_46250030_9_46 |46250030   |104319   |
|ES/SP_46250043_14_6 |46250043   |111335   |
|ES/SP_46250043_8_8  |46250043   |107104   |
|ES/SP_46250046_10_46|46250046   |113090   |
|ES/SP_462

+----------+-----------+---------------+-------------+
|id_tiempo |id_estacion|id_contaminante|concentracion|
+----------+-----------+---------------+-------------+
|2013011820|46250048   |7              |40.0         |
|2013020813|46250048   |7              |53.0         |
|2013021506|46250048   |7              |1.0          |
|2013022317|46250048   |7              |69.0         |
|2013022611|46250048   |7              |38.0         |
|2013030205|46250048   |7              |2.0          |
|2013030310|46250048   |7              |31.0         |
|2013032118|46250048   |7              |42.0         |
|2013032221|46250048   |7              |62.0         |
|2013040318|46250048   |7              |58.0         |
|2013040621|46250048   |7              |74.0         |
|2013041720|46250048   |7              |62.0         |
|2013042905|46250048   |7              |44.0         |
|2013043002|46250048   |7              |9.0          |
|2013050414|46250048   |7              |69.0         |
|201305090

In [28]:
# Comprobamos que todas las mediciones tengan estacion
spark.sql(f"""
    SELECT COUNT(*)                                                AS registros,
           SUM(CASE WHEN e.id_estacion IS NULL THEN 1 ELSE 0 END)  AS sin_estacion,
           SUM(CASE WHEN t.id_tiempo   IS NULL THEN 1 ELSE 0 END)  AS sin_tiempo
    FROM {tabla("fact_aire")} a
    LEFT JOIN {tabla("dim_estacion")} e ON a.id_estacion = e.id_estacion
    LEFT JOIN {tabla("dim_tiempo")}   t ON a.id_tiempo   = t.id_tiempo
""").show()

+---------+------------+----------+
|registros|sin_estacion|sin_tiempo|
+---------+------------+----------+
|  2215571|           0|         0|
+---------+------------+----------+



## 6. Carga de fact_meteo

In [32]:
# Creamos vista para cargar los datos desde el fichero parquet
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW aire_gva
    USING PARQUET
    OPTIONS (path '{RUTA_METEO}')
""")


# Carga de fact_meteo
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("fact_meteo")}
    SELECT
       CAST(DATE_FORMAT(datetime, 'yyyyMMddHH') AS INT) AS id_tiempo,
       station_code AS id_estacion,
       temperature_clean AS temperatura,
       relative_humidity_clean AS humedad_relativa,
       wind_speed_clean AS velocidad_viento,
       wind_direction_deg_clean AS direccion_viento,
       precipitation_clean AS precipitacion,
       pressure_clean AS presion,
       solar_radiation_clean AS radiation_solar
    FROM aire_gva
    WHERE COALESCE(temperature_clean, relative_humidity_clean, wind_speed_clean,
                   wind_direction_deg_clean, precipitation_clean, pressure_clean,
                   solar_radiation_clean) IS NOT NULL
""")

spark.sql(f"SELECT * FROM {tabla('fact_meteo')}").show(truncate=False)

+----------+-----------+-----------+----------------+----------------+----------------+------------+-------+---------------+
|id_tiempo |id_estacion|temperatura|humedad_relativa|velocidad_viento|direccion_viento|precipitacon|presion|radiacion_solar|
+----------+-----------+-----------+----------------+----------------+----------------+------------+-------+---------------+
|2022070100|46250047   |null       |null            |1.3             |68.0            |null        |null   |null           |
|2022070101|46250047   |null       |null            |0.5             |41.0            |null        |null   |null           |
|2022070102|46250047   |null       |null            |0.6             |29.0            |null        |null   |null           |
|2022070103|46250047   |null       |null            |0.4             |35.0            |null        |null   |null           |
|2022070104|46250047   |null       |null            |0.3             |22.0            |null        |null   |null           |


In [33]:
spark.sql(f"""
    SELECT m.id_estacion, e.nombre_estacion, COUNT(*) AS registros,
           MIN(t.fecha) AS desde, MAX(t.fecha) AS hasta
    FROM {tabla("fact_meteo")} m
    LEFT JOIN {tabla("dim_estacion")} e ON m.id_estacion = e.id_estacion
    LEFT JOIN {tabla("dim_tiempo")}   t ON m.id_tiempo   = t.id_tiempo
    GROUP BY m.id_estacion, e.nombre_estacion
    ORDER BY m.id_estacion
""").show(truncate=False)

+-----------+------------------+---------+----------+----------+
|id_estacion|nombre_estacion   |registros|desde     |hasta     |
+-----------+------------------+---------+----------+----------+
|46250047   |AVD. FRANCIA      |51462    |2020-01-01|2025-12-31|
|46250049   |CONSELLERIA METEO.|51259    |2020-01-01|2025-12-31|
|46250900   |NAZARET MET-2     |47886    |2020-05-01|2025-12-31|
+-----------+------------------+---------+----------+----------+



## 7. Carga de fact_trafico

In [35]:
# fact_trafico
# Creamos vista para cargar los datos desde el fichero parquet
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW trafico
    USING PARQUET
    OPTIONS (path '{RUTA_TRAFICO}')
""")


# Carga de fact_meteo
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("fact_trafico")}
    SELECT
       CAST(DATE_FORMAT(fecha_inicio, 'yyyyMMddHH') AS INT) AS id_tiempo,
       CAST(id_espira AS STRING) AS id_espira,
       intensidad
    FROM trafico
    
""")

spark.sql(f"SELECT * FROM {tabla('fact_trafico')}").show(truncate=False)

+----------+---------+----------+
|id_tiempo |id_espira|intensidad|
+----------+---------+----------+
|2024012600|829      |63.0      |
|2024012722|3210     |0.0       |
|2024010410|3210     |293.0     |
|2024010416|3606     |1620.0    |
|2024011310|1436     |627.0     |
|2024010422|1436     |660.0     |
|2024011409|2904     |0.0       |
|2024012616|829      |63.0      |
|2024011102|3606     |126.0     |
|2024011123|3210     |0.0       |
|2024010418|829      |599.0     |
|2024012613|3210     |0.0       |
|2024011109|3606     |3270.0    |
|2024011300|2904     |0.0       |
|2024011310|3606     |1925.0    |
|2024012623|3606     |810.0     |
|2024010108|3606     |209.0     |
|2024012407|3606     |810.0     |
|2024010401|1436     |168.0     |
|2024011411|3606     |1548.0    |
+----------+---------+----------+
only showing top 20 rows



In [36]:
spark.sql(f"""
    SELECT COUNT(*)                                               AS registros,
           COUNT(DISTINCT id_espira)                              AS espiras,
           SUM(CASE WHEN t.id_tiempo IS NULL THEN 1 ELSE 0 END)   AS sin_tiempo
    FROM {tabla("fact_trafico")} f
    LEFT JOIN {tabla("dim_tiempo")} t ON f.id_tiempo = t.id_tiempo
""").show()

+---------+-------+----------+
|registros|espiras|sin_tiempo|
+---------+-------+----------+
| 11650206|    965|         0|
+---------+-------+----------+



Agregar carga prediccion
Agregar carga de las tablas de verde y de ruido cuando las tengamos limpias

## Control de cantidad de datos cargados

In [39]:
spark.sql(f"""
SELECT count(*) from {tabla("fact_trafico")}
union all
SELECT count(*) from {tabla("fact_meteo")}
union all
SELECT count(*) from {tabla("fact_aire")}
union all
SELECT count(*) from {tabla("dim_tiempo")}
union all
SELECT count(*) from {tabla("dim_contaminante")}
union all
SELECT count(*) from {tabla("dim_estacion")}
""").show()



+--------+
|count(1)|
+--------+
|11650206|
|  150607|
| 2215571|
|  131472|
|       4|
|      11|
+--------+

